# Лабораторная работа №12: Генерация синтетического датасета для VLM

**Версия:** финальная (чистовой прогон)  
**Курс:** «Современное компьютерное зрение»

> Сначала выполните notebook в режиме `FAST_DEV_RUN=True`, затем запускайте полный эксперимент. Сохраняйте метрики и checkpoints в `artifacts/`.

## Цели обучения

После работы студент должен уметь:

- Сгенерировать спецификации сцен и изображения.
- Проверить/отфильтровать VLM и embeddings.
- Сравнить real/synthetic mixtures.


## Теоретическая часть

Корректный pipeline: scene specification → renderer/diffusion → VLM verification → filtering → downstream evaluation. Synthetic test не заменяет реальный test; provenance каждого примера обязателен.


### Математическая постановка

При постоянном бюджете **B = 80** смесь обучающих данных задаётся формулой **D(α) = (1 − α)·D_real + α·D_syn**: доля α **замещает** реальные примеры синтетическими, а не добавляется к ним (α = 0 / 0.25 / 0.5 / 0.75 → 80 / 60 / 40 / 20 real). Качество Q(α) — accuracy линейного классификатора count 2 vs 3 на фиксированном реальном test set, поэтому оценивается перенос счёта геометрических фигур на реальные фото. Дубли синтетики отсеиваются по хешу пикселей; доменная близость real/synthetic дополнительно проверяется косинусной близостью центроидов замороженных признаков ResNet18.


### Материалы

- [BlenderProc](https://github.com/DLR-RM/BlenderProc)
- [Diffusers](https://huggingface.co/docs/diffusers/index)

Ссылки даны как дополнительный материал. При изменении API сверяйтесь с актуальной официальной документацией и фиксируйте версии зависимостей.


## Практический протокол

В полном режиме обязательны реальный test-корпус, реальная VLM-проверка синтетики и обучение нескольких downstream-моделей. Процедурный renderer позволяет воспроизводить provenance, но синтетические изображения никогда не называются «реальными». Реальные фотографии и опубликованные метки 2/3 объектов берутся из человечески проверенных validation/test PixMo-Count (ODC-BY-1.0 для датасета; права исходных Flickr-фотографий могут отличаться). Ноутбук автоматически скачивает subset и записывает `data/lab12/real_manifest.jsonl`. Это междоменная оценка переноса счёта геометрических фигур на реальные объекты, а не тест распознавания кругов/квадратов. FAST_DEV_RUN проверяет только renderer и инварианты, не создаёт метрики качества.

In [1]:
from pathlib import Path
import json, os, random, time
import numpy as np
SEED = 42
FAST_DEV_RUN = False  # True: проверка механики на малой выборке, НЕ отчётный эксперимент
ARTIFACTS = Path('artifacts'); ARTIFACTS.mkdir(exist_ok=True)
random.seed(SEED); np.random.seed(SEED)
def save_json(name, data):
    (ARTIFACTS/name).write_text(json.dumps(data, ensure_ascii=False, indent=2, default=str), encoding="utf-8")

VLM_LOADER = "\n# Одна и та же квантованная модель и интерфейс используются для сравниваемых запусков.\n# В Colab предварительно: %pip install -q 'transformers>=4.51,<5' accelerate bitsandbytes pillow datasets\nimport gc\nimport torch\nfrom PIL import Image\nfrom transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration, BitsAndBytesConfig\nMODEL_ID = os.getenv('CV_VLM_MODEL', 'Qwen/Qwen2.5-VL-3B-Instruct')\nassert torch.cuda.is_available(), 'Требуется GPU для реального VLM-инференса; CPU smoke проверяет только данные и код.'\n# Диагностика: квантованной модели нужно ~2.5 ГБ свободной памяти именно на GPU,\n# а bitsandbytes 4-bit не умеет CPU: 'auto' при нехватке VRAM кладёт часть модулей на CPU -> ValueError.\nfree_b, total_b = torch.cuda.mem_get_info()\nprint(f'GPU: {torch.cuda.get_device_name(0)} | VRAM {total_b/2**30:.1f} ГБ, свободно {free_b/2**30:.1f} ГБ')\nif free_b < int(2.6 * 2**30):\n    print('ВНИМАНИЕ: свободной VRAM мало. Закройте браузер/игры и СТАРЫЕ ядра Jupyter (в них может висеть прежняя копия модели), лучше всего перезагрузите ПК.')\ngc.collect(); torch.cuda.empty_cache()\nquant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16)\nprocessor = AutoProcessor.from_pretrained(MODEL_ID)\nmodel = None\nfor _attempt in (1, 2):\n    try:\n        model = Qwen2_5_VLForConditionalGeneration.from_pretrained(MODEL_ID, quantization_config=quant, device_map={'': 0}).eval()\n        break\n    except Exception as _e:\n        model = None\n        print(f'Попытка {_attempt}: модель не разместилась на GPU ({type(_e).__name__}). Чищу кэш и пробую ещё раз...')\n        gc.collect(); torch.cuda.empty_cache()\nif model is None:\n    raise RuntimeError('Модель не поместилась в VRAM даже после очистки кэша. Освободите GPU: закройте браузер/игры, shut down старых ядер Jupyter (кто занял VRAM: nvidia-smi), лучше всего перезагрузите ПК, затем Kernel -> Restart -> Run All.')\ndef infer_image(path, prompt, adapter=None):\n    messages = [{'role':'user','content':[{'type':'image','image':Image.open(path).convert('RGB')},{'type':'text','text':prompt}]}]\n    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)\n    inputs = processor(text=[text], images=[Image.open(path).convert('RGB')], return_tensors='pt').to(model.device)\n    with torch.no_grad(): generated = model.generate(**inputs, max_new_tokens=96, do_sample=False)\n    return processor.batch_decode(generated[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0].strip()\n"

In [2]:
# Сеть: если HTTPS перехватывается прокси/антивирусом (self-signed сертификат),
# переводим проверку сертификатов на системное хранилище ОС (пакет truststore,
# однократно: pip install truststore). Если пакета нет — стандартная проверка.
try:
    import truststore, urllib3
    truststore.inject_into_ssl()
    urllib3.disable_warnings()
    print("SSL: проверка сертификатов через системное хранилище ОС (truststore)")
except ImportError:
    print("SSL: truststore не установлен — стандартная проверка сертификатов")


SSL: проверка сертификатов через системное хранилище ОС (truststore)


## Оценивание

Десять обязательных предметных этапов — по 1 баллу каждый, всего 10 баллов. Отдельное творческое исследование — до 1 дополнительного балла. Каждый этап принимается по указанному наблюдаемому результату.

### Геометрический рендерер

*Вес: 1 балл.*

**Постановка.** Этап «геометрический рендерер» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Реализуйте рендерер цветных фигур без перекрытий с фиксированными координатами.
2. Сохраните и предъявите функция render и словарь палитры.

**Результат.** функция render и словарь палитры.

**Критерий принятия.** Задание считается принятым, если функция render и словарь палитры получены из фактических входов этого этапа и доступны для проверки.

In [3]:
# ЭТАП 1 — геометрический рендерер: PIL-отрисовка цветных фигур без перекрытий.
# Координаты, формы и цвета фиксированы в спецификации сцены (ЭТАП 2);
# render только отрисовывает и проверяет инвариант неперекрытия.
from PIL import Image, ImageDraw

PALETTE = {  # имя цвета -> RGB
    "red": (214, 69, 65), "blue": (46, 110, 208), "green": (52, 152, 91),
    "yellow": (241, 196, 60), "purple": (125, 74, 178), "orange": (230, 126, 44),
}
SHAPES = ("circle", "square", "triangle")
BG = (245, 245, 245)   # светлый однотонный фон
SS = 2                 # суперсэмплинг: рисуем в SS раз крупнее, затем уменьшаем (гладкие края)

def render(spec, size=256):
    """Рисует сцену по спецификации. spec: {"id", "objects": [{shape, color, center, size}]}."""
    img = Image.new("RGB", (size * SS, size * SS), BG)
    dr = ImageDraw.Draw(img)
    objs = spec["objects"]
    # инвариант: дистанция центров >= r1 + r2 + зазор (перекрытий нет по построению)
    for i in range(len(objs)):
        for j in range(i + 1, len(objs)):
            a, b = objs[i], objs[j]
            d = ((a["center"][0] - b["center"][0]) ** 2 + (a["center"][1] - b["center"][1]) ** 2) ** 0.5
            assert d >= (a["size"] + b["size"]) / 2 + 12, f"перекрытие в {spec.get('id', '?')}: {a} vs {b}"
    for o in objs:
        cx, cy = o["center"][0] * SS, o["center"][1] * SS
        r = (o["size"] / 2) * SS
        color = PALETTE[o["color"]]
        outline = tuple(int(c * 0.55) for c in color)
        if o["shape"] == "circle":
            dr.ellipse([cx - r, cy - r, cx + r, cy + r], fill=color, outline=outline, width=2 * SS)
        elif o["shape"] == "square":
            dr.rectangle([cx - r, cy - r, cx + r, cy + r], fill=color, outline=outline, width=2 * SS)
        elif o["shape"] == "triangle":
            dr.polygon([(cx, cy - r), (cx - r * 0.9, cy + r * 0.75), (cx + r * 0.9, cy + r * 0.75)],
                       fill=color, outline=outline)
        else:
            raise ValueError(f"неизвестная фигура: {o['shape']}")
    return img.resize((size, size), Image.LANCZOS)

# демо-сцена (артефакт этапа: функция render + словарь палитры; отрисовка доступна для проверки)
_demo = {"id": "demo", "objects": [
    {"shape": "circle", "color": "red", "center": [78, 120], "size": 56},
    {"shape": "square", "color": "blue", "center": [158, 84], "size": 48},
    {"shape": "triangle", "color": "green", "center": [166, 172], "size": 56},
]}
render(_demo).save(ARTIFACTS / "render_demo.png")
print("ЭТАП 1 OK: render() + PALETTE готовы; инвариант неперекрытия встроен в render;")
print("демо-сцена из", len(_demo["objects"]), "фигур -> artifacts/render_demo.png; цвета:", list(PALETTE))


ЭТАП 1 OK: render() + PALETTE готовы; инвариант неперекрытия встроен в render;
демо-сцена из 3 фигур -> artifacts/render_demo.png; цвета: ['red', 'blue', 'green', 'yellow', 'purple', 'orange']


### Пространство визуальных сцен

*Вес: 1 балл.*

**Постановка.** Этап «пространство визуальных сцен» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Сгенерируйте воспроизводимые сочетания формы, цвета и числа объектов.
2. Сохраните и предъявите воспроизводимые спецификации фигур и количества.

**Результат.** воспроизводимые спецификации фигур и количества.

**Критерий принятия.** Задание считается принятым, если воспроизводимые спецификации фигур и количества получены из фактических входов этого этапа и доступны для проверки.

In [4]:
# ЭТАП 2 — пространство визуальных сцен: воспроизводимые спецификации
# (форма, цвет, число объектов 1..3, координаты без перекрытий) при фиксированном SEED.
from collections import Counter

N_SYN = 6 if FAST_DEV_RUN else 180   # сколько сцен генерируем
MARGIN = 34                          # минимальная дистанция центра фигуры от края
R_RANGE = (26, 38)                   # радиус фигуры, px

rng_scene = random.Random(SEED)      # отдельный генератор: спецификации воспроизводимы

def make_spec(idx):
    count = rng_scene.choice((1, 2, 3))
    objects = []
    for _ in range(count):
        for _attempt in range(300):
            r = rng_scene.randint(*R_RANGE)
            c = [rng_scene.randint(MARGIN + r, 256 - MARGIN - r),
                 rng_scene.randint(MARGIN + r, 256 - MARGIN - r)]
            if all(((c[0] - o["center"][0]) ** 2 + (c[1] - o["center"][1]) ** 2) ** 0.5
                   >= (r + o["size"] / 2) + 12 for o in objects):
                objects.append({"shape": rng_scene.choice(SHAPES), "color": rng_scene.choice(list(PALETTE)),
                                "center": c, "size": 2 * r})
                break
        else:
            raise RuntimeError("не удалось разместить фигуру без перекрытий")
    return {"id": f"syn-{idx:04d}", "count": count, "objects": objects}

specs = [make_spec(i) for i in range(1, N_SYN + 1)]
print(f"ЭТАП 2 OK: {len(specs)} спецификаций, распределение count = {dict(Counter(s['count'] for s in specs))}")
print("повторный запуск при том же SEED даёт те же спецификации; неперекрытие дополнительно проверяет render()")


ЭТАП 2 OK: 180 спецификаций, распределение count = {3: 62, 2: 55, 1: 63}
повторный запуск при том же SEED даёт те же спецификации; неперекрытие дополнительно проверяет render()


### Изображения и вопросы

*Вес: 1 балл.*

**Постановка.** Этап «изображения и вопросы» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Сохраните изображения, вопрос и ответ для каждой сцены.
2. Сохраните и предъявите файлы изображений и связанные с ними ответы.

**Результат.** файлы изображений и связанные с ними ответы.

**Критерий принятия.** Задание считается принятым, если файлы изображений и связанные с ними ответы получены из фактических входов этого этапа и доступны для проверки.

In [5]:
# ЭТАП 3 — изображения и вопросы: рендер каждой спецификации в PNG,
# для каждой сцены — вопрос и ответ (ground truth известен из спецификации).
SYN_DIR = Path("data/lab12/synthetic"); SYN_DIR.mkdir(parents=True, exist_ok=True)
SYN_QUESTION = "How many shapes are in the image? Answer with one integer only."

t0 = time.time()
syn_rows = []
for s in specs:
    path = SYN_DIR / f"{s['id']}.png"
    render(s).save(path)
    syn_rows.append({"id": s["id"], "image": str(path), "question": SYN_QUESTION,
                     "answer": str(s["count"]), "count": s["count"]})
(ARTIFACTS / "synthetic_qa.jsonl").write_text(
    "".join(json.dumps(r, ensure_ascii=False) + "\n" for r in syn_rows), encoding="utf-8")
print(f"ЭТАП 3 OK: {len(syn_rows)} изображений -> {SYN_DIR} ({time.time() - t0:.1f} c); вопросы/ответы -> artifacts/synthetic_qa.jsonl")
print("пример:", json.dumps(syn_rows[0], ensure_ascii=False))


ЭТАП 3 OK: 180 изображений -> data\lab12\synthetic (1.0 c); вопросы/ответы -> artifacts/synthetic_qa.jsonl
пример: {"id": "syn-0001", "image": "data\\lab12\\synthetic\\syn-0001.png", "question": "How many shapes are in the image? Answer with one integer only.", "answer": "3", "count": 3}


### Происхождение синтетических данных

*Вес: 1 балл.*

**Постановка.** Этап «происхождение синтетических данных» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Зафиксируйте seed, спецификацию, пути и provenance в манифесте.
2. Сохраните и предъявите artifacts/synthetic_manifest.jsonl.

**Результат.** artifacts/synthetic_manifest.jsonl.

**Критерий принятия.** Задание считается принятым, если artifacts/synthetic_manifest.jsonl получены из фактических входов этого этапа и доступны для проверки.

In [6]:
# ЭТАП 4 — происхождение синтетических данных: seed, полная спецификация, пути и
# версии инструментов фиксируются в манифесте; каждый пример воспроизводим из записи.
from datetime import datetime, timezone

manifest = []
for s, r in zip(specs, syn_rows):
    manifest.append({
        "id": s["id"], "image": r["image"], "question": r["question"], "answer": r["answer"],
        "count": s["count"], "seed": SEED, "spec": s,
        "renderer": "PIL.ImageDraw; суперсэмплинг x2; фон (245,245,245); палитра 6 цветов",
        "palette_version": "1.0", "generator": "make_spec (ЭТАП 2), random.Random(SEED)",
        "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    })
(ARTIFACTS / "synthetic_manifest.jsonl").write_text(
    "".join(json.dumps(m, ensure_ascii=False) + "\n" for m in manifest), encoding="utf-8")

# проверка: манифест читается обратно, ключевые поля сходятся со спецификациями
_back = [json.loads(x) for x in (ARTIFACTS / "synthetic_manifest.jsonl").read_text(encoding="utf-8").splitlines()]
assert len(_back) == len(specs) and all(b["id"] == s["id"] and b["count"] == s["count"]
                                        for b, s in zip(_back, specs))
print(f"ЭТАП 4 OK: artifacts/synthetic_manifest.jsonl, {len(manifest)} записей (seed={SEED}); обратное чтение сходится")


ЭТАП 4 OK: artifacts/synthetic_manifest.jsonl, 180 записей (seed=42); обратное чтение сходится


### Ответы VLM и визуальная дедупликация

*Вес: 1 балл.*

**Постановка.** Этап «ответы vlm и визуальная дедупликация» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Спросите реальную VLM о числе фигур и найдите дубли по хешу пикселей.
2. Сохраните и предъявите artifacts/verification.jsonl с реальными ответами VLM.

**Результат.** artifacts/verification.jsonl с реальными ответами VLM.

**Критерий принятия.** Задание считается принятым, если artifacts/verification.jsonl с реальными ответами VLM получены из фактических входов этого этапа и доступны для проверки.

In [7]:
# ЭТАП 5 — ответы реальной VLM о числе фигур + визуальная дедупликация по хешу пикселей.
# FAST_DEV_RUN по условию проверяет только renderer и инварианты — VLM-проверка пропускается.
import hashlib, re

if FAST_DEV_RUN:
    verification = []
    print("FAST_DEV_RUN: VLM-проверка пропущена (в быстром режиме метрики качества не создаются)")
else:
    exec(VLM_LOADER)  # квантованная Qwen2.5-VL-3B-Instruct: processor, model, infer_image
    WORD_NUM = {"one": 1, "two": 2, "three": 3, "four": 4, "five": 5}

    def parse_count(text):
        m = re.search(r"\d+", text)
        if m:
            return int(m.group())
        for w, n in WORD_NUM.items():
            if w in text.lower():
                return n
        return None

    def pixel_sha(path):
        with Image.open(path) as im:
            return hashlib.sha256(im.convert("RGB").tobytes()).hexdigest()

    verification, seen_pixels = [], {}
    t0 = time.time()
    for i, r in enumerate(syn_rows, 1):
        ts = time.time()
        raw = infer_image(r["image"], r["question"])
        vlm_count = parse_count(raw)
        ph = pixel_sha(r["image"])
        dup_of = seen_pixels.get(ph)
        if dup_of is None:
            seen_pixels[ph] = r["id"]
        verification.append({"id": r["id"], "image": r["image"], "true_count": r["count"],
                             "vlm_raw": raw, "vlm_count": vlm_count, "agree": vlm_count == r["count"],
                             "pixel_sha256": ph, "dup_of": dup_of, "elapsed_s": round(time.time() - ts, 2)})
        if i % 20 == 0 or i == len(syn_rows):
            rate = sum(v["agree"] for v in verification) / len(verification)
            print(f"[VLM] {i}/{len(syn_rows)} | согласие {rate:.1%} | {time.time() - t0:.0f} c")

    (ARTIFACTS / "verification.jsonl").write_text(
        "".join(json.dumps(v, ensure_ascii=False) + "\n" for v in verification), encoding="utf-8")
    n_agree = sum(v["agree"] for v in verification)
    n_dup = sum(v["dup_of"] is not None for v in verification)
    print(f"ЭТАП 5 OK: artifacts/verification.jsonl — {len(verification)} сцен, согласие {n_agree} "
          f"({n_agree / len(verification):.1%}), дублей по пиксельному хешу {n_dup}, время {time.time() - t0:.0f} c")


W1028 13:41:09.415000 6676 site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1028 13:41:09.523000 6676 site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


GPU: NVIDIA GeForce GTX 1050 Ti | VRAM 4.0 ГБ, свободно 3.3 ГБ


The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
c:\Users\eric1\.conda\envs\fmi-cv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:957: UserWarning: inner dimension (3420) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(


[VLM] 20/180 | согласие 100.0% | 33 c
[VLM] 40/180 | согласие 95.0% | 64 c
[VLM] 60/180 | согласие 95.0% | 96 c
[VLM] 80/180 | согласие 95.0% | 128 c
[VLM] 100/180 | согласие 94.0% | 159 c
[VLM] 120/180 | согласие 95.0% | 191 c
[VLM] 140/180 | согласие 94.3% | 223 c
[VLM] 160/180 | согласие 95.0% | 254 c
[VLM] 180/180 | согласие 95.0% | 286 c
ЭТАП 5 OK: artifacts/verification.jsonl — 180 сцен, согласие 171 (95.0%), дублей по пиксельному хешу 0, время 286 c


### Отбор согласованных пар

*Вес: 1 балл.*

**Постановка.** Этап «отбор согласованных пар» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Отберите только изображения с согласованным ответом и без дублей.
2. Сохраните и предъявите artifacts/accepted.jsonl после проверки ответов.

**Результат.** artifacts/accepted.jsonl после проверки ответов.

**Критерий принятия.** Задание считается принятым, если artifacts/accepted.jsonl после проверки ответов получены из фактических входов этого этапа и доступны для проверки.

In [8]:
# ЭТАП 6 — отбор согласованных пар: ответ VLM совпал с истинным числом И сцена не дубль.
from collections import Counter

accepted = [dict(v, question=r["question"], answer=r["answer"])
            for v, r in zip(verification, syn_rows) if v["agree"] and v["dup_of"] is None]
(ARTIFACTS / "accepted.jsonl").write_text(
    "".join(json.dumps(a, ensure_ascii=False) + "\n" for a in accepted), encoding="utf-8")

if verification:
    by_count = Counter(a["true_count"] for a in accepted)
    print(f"ЭТАП 6 OK: artifacts/accepted.jsonl — {len(accepted)} сцен (по count: {dict(sorted(by_count.items()))}); "
          f"отсеяно: несогласных {sum(not v['agree'] for v in verification)}, "
          f"дублей {sum(v['dup_of'] is not None for v in verification)}")
else:
    print("ЭТАП 6: FAST_DEV_RUN — accepted.jsonl пуст (проверка механики)")


ЭТАП 6 OK: artifacts/accepted.jsonl — 171 сцен (по count: {1: 63, 2: 55, 3: 53}); отсеяно: несогласных 9, дублей 0


### Независимый набор реальных изображений

*Вес: 1 балл.*

**Постановка.** Этап «независимый набор реальных изображений» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Скачайте PixMo-Count validation для обучения и официальный test для отложенной оценки; проверьте метки и отсутствие общих ID/изображений.
2. Сохраните и предъявите проверенный real train/test manifest с источником, split и контрольными суммами.

**Результат.** проверенный real train/test manifest с источником, split и контрольными суммами.

**Критерий принятия.** Задание считается принятым, если проверенный real train/test manifest получены из фактических входов этого этапа и доступны для проверки.

In [9]:
# ЭТАП 7 — независимый набор реальных изображений: PixMo-Count (ODC-BY-1.0),
# validation -> real train, официальный test -> отложенная оценка. Логика загрузки —
# по materials/open_real_scenes.py, дополнена exclude_ids (для бонусной выборки).
import hashlib
import urllib.request
from urllib.parse import urlencode

PIXMO_API = 'https://datasets-server.huggingface.co/rows'

def load_real_scenes(root='data/lab12', train_size=80, test_size=12, exclude_ids=()):
    """Скачивает реальные фото с человечески проверенными метками count 2/3;
    пишет манифест с источником, сплитом и контрольными суммами."""
    from PIL import Image as _Image
    root = Path(root); root.mkdir(parents=True, exist_ok=True)
    exclude = set(exclude_ids)
    out = []
    for split, source_split, limit in [('train', 'validation', train_size), ('test', 'test', test_size)]:
        chosen = []
        for offset in range(0, 540, 100):
            query = urlencode(dict(dataset='allenai/pixmo-count', config='default', split=source_split,
                                   offset=offset, length=min(100, 540 - offset)))
            with urllib.request.urlopen(PIXMO_API + '?' + query, timeout=35) as response:
                rows = json.load(response)['rows']
            for entry in rows:
                r = entry['row']
                pid = f'pixmo-count-{split}-{entry["row_idx"]}'
                if pid in exclude:
                    continue
                if r['count'] not in (2, 3) or sum(v['answer'] == str(r['count']) for v in chosen) >= (limit + 1) // 2:
                    continue
                path = root / f'{pid}.jpg'
                try:
                    if not path.exists():
                        req = urllib.request.Request(r['image_url'],
                                                     headers={'User-Agent': 'Mozilla/5.0 (educational research)'})
                        with urllib.request.urlopen(req, timeout=30) as response:
                            data = response.read()
                        path.write_bytes(data)
                    with _Image.open(path) as image:
                        image.verify()
                except Exception:
                    path.unlink(missing_ok=True)
                    continue
                chosen.append(dict(id=pid, split=split, image=str(path.resolve()), answer=str(r['count']),
                                   label=r['label'],
                                   question=f"How many {r['label']} are visible? Answer with one integer.",
                                   source='https://huggingface.co/datasets/allenai/pixmo-count',
                                   source_row=entry['row_idx'], source_split=source_split,
                                   image_url=r['image_url'], image_sha256=r['image_sha256'],
                                   downloaded_sha256=hashlib.sha256(path.read_bytes()).hexdigest(),
                                   checksum_matches=hashlib.sha256(path.read_bytes()).hexdigest() == r['image_sha256'],
                                   dataset_license='ODC-BY-1.0; image rights may differ'))
                if len(chosen) >= limit:
                    break
            if len(chosen) >= limit:
                break
        if limit and (len(chosen) < limit or {r['answer'] for r in chosen} != {'2', '3'}):
            raise RuntimeError(f'{split}: {len(chosen)}/{limit} доступных изображений или неполный класс; '
                               f'перезапустите ячейку — скачанные файлы кэшируются')
        out.extend(chosen)
    if test_size:
        # пересечение проверяется и по эталонному хешу, и по фактически скачанным байтам
        if ({r['image_sha256'] for r in out[:train_size]} & {r['image_sha256'] for r in out[train_size:]}
                or {r['downloaded_sha256'] for r in out[:train_size]} & {r['downloaded_sha256'] for r in out[train_size:]}):
            raise ValueError('пересечение real train/test')
    (root / 'real_manifest.jsonl').write_text(''.join(json.dumps(x, ensure_ascii=False) + '\n' for x in out),
                                              encoding='utf-8')
    return out

if FAST_DEV_RUN:
    real_rows = []
    print("FAST_DEV_RUN: загрузка реальных изображений пропущена (проверяется только синтетическая механика)")
else:
    t0 = time.time()
    real_rows = load_real_scenes(train_size=80, test_size=12)
    tr = [r for r in real_rows if r['split'] == 'train']
    te = [r for r in real_rows if r['split'] == 'test']
    ok_sum = sum(r['checksum_matches'] for r in real_rows)
    print(f"ЭТАП 7 OK: real train {len(tr)} (2:{sum(r['answer'] == '2' for r in tr)} / 3:{sum(r['answer'] == '3' for r in tr)}), "
          f"test {len(te)} (2:{sum(r['answer'] == '2' for r in te)} / 3:{sum(r['answer'] == '3' for r in te)}); "
          f"контрольные суммы сошлись {ok_sum}/{len(real_rows)} — расхождения фиксируются в манифесте "
          f"(хостинг может отдавать другой размер файла; human-проверенные метки count не затронуты); "
          f"{time.time() - t0:.0f} c -> data/lab12/real_manifest.jsonl")


ЭТАП 7 OK: real train 80 (2:40 / 3:40), test 12 (2:6 / 3:6); контрольные суммы сошлись 12/92 — расхождения фиксируются в манифесте (хостинг может отдавать другой размер файла; human-проверенные метки count не затронуты); 36 c -> data/lab12/real_manifest.jsonl


### Признаки реальных и синтетических сцен

*Вес: 1 балл.*

**Постановка.** Этап «признаки реальных и синтетических сцен» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Вычислите одинаковые замороженные признаки ResNet18 для real train, synthetic и real test.
2. Сохраните и предъявите замороженные признаки ResNet18 для трёх наборов.

**Результат.** замороженные признаки ResNet18 для трёх наборов.

**Критерий принятия.** Задание считается принятым, если замороженные признаки ResNet18 для трёх наборов получены из фактических входов этого этапа и доступны для проверки.

In [10]:
# ЭТАП 8 — одинаковые замороженные признаки ResNet18 (ImageNet-веса, без fine-tune)
# для real train, accepted-синтетики (классы 2/3, общие с real) и real test.
# Если torchvision не установлен: %pip install -q torchvision
if FAST_DEV_RUN:
    print("FAST_DEV_RUN: признаки не вычисляются (в быстром режиме метрики качества не создаются)")
else:
    import torch
    from torchvision.models import resnet18, ResNet18_Weights
    from torchvision import transforms

    _tf = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor(),
                              transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])
    _dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    FEAT_MODEL = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
    FEAT_MODEL.fc = torch.nn.Identity()          # 512-d признаки до классификационной головы
    FEAT_MODEL = FEAT_MODEL.eval().to(_dev)

    def extract_features(rows, tag):
        ids, feats = [], []
        for i, r in enumerate(rows, 1):
            with Image.open(r['image']) as im:
                x = _tf(im.convert('RGB')).unsqueeze(0).to(_dev)
            with torch.no_grad():
                feats.append(FEAT_MODEL(x).squeeze(0).cpu().numpy())
            ids.append(r['id'])
            if i % 40 == 0 or i == len(rows):
                print(f"[features:{tag}] {i}/{len(rows)}")
        return np.stack(feats).astype('float32'), np.array(ids)

    real_train_rows = [r for r in real_rows if r['split'] == 'train']
    real_test_rows = [r for r in real_rows if r['split'] == 'test']
    acc23 = [a for a in accepted if a['true_count'] in (2, 3)]   # классы, общие с real

    X_real_train, ids_real_train = extract_features(real_train_rows, 'real_train')
    X_syn, ids_syn = extract_features(acc23, 'synthetic')
    X_real_test, ids_real_test = extract_features(real_test_rows, 'real_test')
    y_real_train = np.array([int(r['answer']) for r in real_train_rows])
    y_syn = np.array([a['true_count'] for a in acc23])
    y_real_test = np.array([int(r['answer']) for r in real_test_rows])

    np.savez_compressed(ARTIFACTS / 'features_real_train.npz', ids=ids_real_train, X=X_real_train, y=y_real_train)
    np.savez_compressed(ARTIFACTS / 'features_synthetic.npz', ids=ids_syn, X=X_syn, y=y_syn)
    np.savez_compressed(ARTIFACTS / 'features_real_test.npz', ids=ids_real_test, X=X_real_test, y=y_real_test)
    print(f"ЭТАП 8 OK: замороженный ResNet18, признаки 512-d -> real train {X_real_train.shape}, "
          f"synthetic {X_syn.shape}, real test {X_real_test.shape}; artifacts/features_*.npz")


[features:real_train] 40/80
[features:real_train] 80/80
[features:synthetic] 40/108
[features:synthetic] 80/108
[features:synthetic] 108/108
[features:real_test] 12/12
ЭТАП 8 OK: замороженный ResNet18, признаки 512-d -> real train (80, 512), synthetic (108, 512), real test (12, 512); artifacts/features_*.npz


### Обучение на смесях

*Вес: 1 балл.*

**Постановка.** Этап «обучение на смесях» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. При постоянном бюджете train обучите классификаторы с четырьмя долями синтетики.
2. Сохраните и предъявите результаты классификаторов при четырёх долях синтетики.

**Результат.** результаты классификаторов при четырёх долях синтетики.

**Критерий принятия.** Задание считается принятым, если результаты классификаторов при четырёх долях синтетики получены из фактических входов этого этапа и доступны для проверки.

In [11]:
# ЭТАП 9 — обучение на смесях при ПОСТОЯННОМ бюджете B=80: доля синтетики alpha
# ЗАМЕНЯЕТ часть реальных примеров (как в D_alpha=(1-alpha)*D_real+alpha*D_syn):
# alpha=0 -> 80 real; 0.25 -> 60 real + 20 syn; 0.5 -> 40+40; 0.75 -> 20+60.
# Классификатор один и тот же, протокол одинаковый, seed фиксирован.
if FAST_DEV_RUN:
    print("FAST_DEV_RUN: обучение смесей пропущено")
else:
    import torch
    import torch.nn as nn

    BUDGET, N_EPOCHS, LR = 80, 400, 5e-3

    def train_linear(X, y, seed=SEED, epochs=N_EPOCHS, lr=LR):
        """Линейный классификатор (count 2 vs 3) на замороженных признаках; full-batch."""
        torch.manual_seed(seed)
        mu, sd = X.mean(0), X.std(0) + 1e-8
        Xn = torch.tensor((X - mu) / sd)
        yt = torch.tensor((y == 3).astype('int64'))          # класс 1 = count 3
        W = torch.zeros(X.shape[1], 2, requires_grad=True)   # выпуклая задача: init не влияет
        b = torch.zeros(2, requires_grad=True)
        opt = torch.optim.Adam([W, b], lr=lr)
        for _ in range(epochs):
            opt.zero_grad()
            loss = nn.functional.cross_entropy(Xn @ W + b, yt)
            loss.backward()
            opt.step()
        return {"mu": mu, "sd": sd, "W": W.detach().numpy(), "b": b.detach().numpy(),
                "train_acc": float((Xn @ W + b).argmax(1).eq(yt).float().mean())}

    def predict(model, X):
        Xn = (X - model["mu"]) / model["sd"]
        t = torch.tensor(Xn) @ torch.tensor(model["W"]) + torch.tensor(model["b"])
        return np.where(t.argmax(1).numpy() == 1, 3, 2)

    # hold-out синтетики для бонуса: до 20 accepted-сцен не участвуют в смесях
    rng_hold = random.Random(SEED + 7)
    hold_n = min(20, 2 * max(0, (len(acc23) - 70) // 2))
    idx_hold = []
    if hold_n:
        for cls, need in ((2, hold_n // 2), (3, hold_n - hold_n // 2)):
            cand = [i for i in range(len(acc23)) if y_syn[i] == cls]
            rng_hold.shuffle(cand)
            idx_hold += cand[:need]
    idx_pool = [i for i in range(len(acc23)) if i not in set(idx_hold)]
    X_syn_pool, y_syn_pool, ids_syn_pool = X_syn[idx_pool], y_syn[idx_pool], ids_syn[idx_pool]

    def pick_stratified(ids, y, k):
        """k id со стратификацией поровну между классами 2/3 (seed-порядок)."""
        idx = list(range(len(ids))); rng_mix.shuffle(idx)
        take = []
        for cls, need in ((2, k // 2), (3, k - k // 2)):
            take += [i for i in idx if y[i] == cls][:need]
        if len(take) < k:
            take += [i for i in idx if i not in take][:k - len(take)]
        return sorted(take)

    rng_mix = random.Random(SEED)
    alphas = [a for a in (0.0, 0.25, 0.5, 0.75)
              if round(BUDGET * (1 - a)) <= len(X_real_train) and round(BUDGET * a) <= len(X_syn_pool)]
    skipped = [a for a in (0.0, 0.25, 0.5, 0.75) if a not in alphas]
    mixture_models, mixture_info = {}, []
    for a in alphas:
        n_syn = round(BUDGET * a); n_real = BUDGET - n_syn
        ri = pick_stratified(ids_real_train, y_real_train, n_real)
        si = pick_stratified(ids_syn_pool, y_syn_pool, n_syn)
        X = np.concatenate([X_real_train[ri], X_syn_pool[si]])
        y = np.concatenate([y_real_train[ri], y_syn_pool[si]])
        model = train_linear(X, y)
        mixture_models[a] = model
        mixture_info.append({"alpha": a, "n_real": n_real, "n_synthetic": n_syn,
                             "real_ids": [str(ids_real_train[i]) for i in ri],
                             "synthetic_ids": [str(ids_syn_pool[i]) for i in si],
                             "train_acc": round(model["train_acc"], 4)})
        print(f"[смесь alpha={a}] real {n_real} + synthetic {n_syn} | train_acc {model['train_acc']:.3f}")
    if skipped:
        print(f"ВНИМАНИЕ: смеси {skipped} пропущены — accepted-синтетики {len(X_syn_pool)} недостаточно")
    save_json("mixture_training.json", {
        "budget": BUDGET, "epochs": N_EPOCHS, "lr": LR, "alphas": alphas, "skipped_alphas": skipped,
        "synthetic_pool": len(X_syn_pool), "synthetic_holdout": [str(ids_syn[i]) for i in idx_hold],
        "mixtures": mixture_info})
    print(f"ЭТАП 9 OK: обучено {len(mixture_models)} классификаторов при бюджете {BUDGET} "
          f"(hold-out синтетики {hold_n}) -> artifacts/mixture_training.json")


[смесь alpha=0.0] real 80 + synthetic 0 | train_acc 1.000
[смесь alpha=0.25] real 60 + synthetic 20 | train_acc 1.000
[смесь alpha=0.5] real 40 + synthetic 40 | train_acc 1.000
[смесь alpha=0.75] real 20 + synthetic 60 | train_acc 1.000
ЭТАП 9 OK: обучено 4 классификаторов при бюджете 80 (hold-out синтетики 20) -> artifacts/mixture_training.json


### Оценка на реальном тесте

*Вес: 1 балл.*

**Постановка.** Этап «оценка на реальном тесте» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Запишите предсказания и accuracy всех моделей только на реальном test.
2. Сохраните и предъявите mixture_results.json и predictions.jsonl.

**Результат.** mixture_results.json и predictions.jsonl.

**Критерий принятия.** Задание считается принятым, если mixture_results.json и predictions.jsonl получены из фактических входов этого этапа и доступны для проверки.

In [12]:
# ЭТАП 10 — оценка всех смесей ТОЛЬКО на отложенном реальном test.
if FAST_DEV_RUN:
    print("FAST_DEV_RUN: оценка на реальном тесте пропущена (в быстром режиме метрики качества не создаются)")
else:
    preds_rows, results = [], []
    for a in alphas:
        p = predict(mixture_models[a], X_real_test)
        acc = float((p == y_real_test).mean())
        per_class = {str(c): round(float((p[y_real_test == c] == c).mean()), 4)
                     for c in (2, 3) if (y_real_test == c).any()}
        results.append({"alpha": a, "n_real": BUDGET - round(BUDGET * a), "n_synthetic": round(BUDGET * a),
                        "train_acc": round(mixture_models[a]["train_acc"], 4),
                        "test_acc": round(acc, 4), "per_class_acc": per_class})
        preds_rows.append((a, p))
        print(f"[test alpha={a}] acc {acc:.3f} | по классам {per_class}")

    with open(ARTIFACTS / 'predictions.jsonl', 'w', encoding='utf-8') as f:
        for i, rid in enumerate(ids_real_test):
            row = {"id": str(rid), "gold": int(y_real_test[i]),
                   "predictions": {str(a): int(preds_rows[j][1][i]) for j, _ in enumerate(preds_rows)}}
            row["correct"] = {str(a): bool(preds_rows[j][1][i] == y_real_test[i]) for j, _ in enumerate(preds_rows)}
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    majority = max(Counter(y_real_test.tolist()).values()) / len(y_real_test)
    best = max(results, key=lambda r: r['test_acc'])
    save_json("mixture_results.json", {
        "budget": BUDGET, "n_test": len(y_real_test),
        "class_balance": {str(k): v for k, v in Counter(y_real_test.tolist()).items()},
        "majority_baseline": round(majority, 4), "results": results, "best_alpha": best['alpha'],
        "note": "test n=12: шаг точности 1/12≈0.083; различия между смесями < 0.17 в пределах шума выборки"})
    print(f"ЭТАП 10 OK: artifacts/mixture_results.json + artifacts/predictions.jsonl; "
          f"лучшая alpha={best['alpha']} (acc {best['test_acc']}), majority baseline {majority:.3f}")


[test alpha=0.0] acc 0.333 | по классам {'2': 0.1667, '3': 0.5}
[test alpha=0.25] acc 0.417 | по классам {'2': 0.5, '3': 0.3333}
[test alpha=0.5] acc 0.667 | по классам {'2': 0.5, '3': 0.8333}
[test alpha=0.75] acc 0.500 | по классам {'2': 0.3333, '3': 0.6667}
ЭТАП 10 OK: artifacts/mixture_results.json + artifacts/predictions.jsonl; лучшая alpha=0.5 (acc 0.6667), majority baseline 0.500


### Творческое мини-исследование

*Бонус: до 1 балла сверх 10.*

**Постановка.** Сформулируйте проверяемую гипотезу о предметном ограничении метода данной лабораторной и проверьте её на дополнительных реальных входах без использования отложенного теста для выбора гипотезы. Сопоставьте результаты с основным экспериментом и объясните расхождения.

**Результат.** Отдельный файл artifacts/bonus.md с гипотезой, протоколом, наблюдениями и выводом.

**Критерий принятия.** Задание считается принятым, если гипотеза проверена по сохранённым предсказаниям или изображениям и приведено воспроизводимое сравнение.

In [13]:
# БОНУС — гипотеза (сформулирована ДО прогона на дополнительных реальных входах):
# эффект смесей real/synthetic, измеренный на отложенном test (n=12), не воспроизводится
# на независимой реальной выборке bonus_val того же распределения — наблюдаемые на test
# различия определяются шумом малой выборки, а не реальным вкладом синтетики.
# Дополнительно: доменный зазор синтетика->real в признаках ResNet18.
if FAST_DEV_RUN:
    print("FAST_DEV_RUN: бонус пропущен")
else:
    t0 = time.time()
    bonus_rows = load_real_scenes(root='data/lab12/bonus', train_size=16, test_size=0,
                                  exclude_ids=[r['id'] for r in real_rows])
    (ARTIFACTS / 'bonus_real_manifest.jsonl').write_text(
        ''.join(json.dumps(x, ensure_ascii=False) + '\n' for x in bonus_rows), encoding='utf-8')
    X_bonus, ids_bonus = extract_features(bonus_rows, 'bonus')
    y_bonus = np.array([int(r['answer']) for r in bonus_rows])

    # (1) те же смеси на bonus_val: воспроизводится ли ранжирование с test?
    bonus_results = [{"alpha": a, "bonus_acc": round(float((predict(mixture_models[a], X_bonus) == y_bonus).mean()), 4)}
                     for a in alphas]
    best_test = max(results, key=lambda r: r['test_acc'])['alpha']
    best_bonus = max(bonus_results, key=lambda r: r['bonus_acc'])['alpha']
    replicated = best_test == best_bonus

    # (2) синтетика-only: in-domain (hold-out синтетики) против cross-domain (bonus real)
    model_syn = train_linear(X_syn_pool, y_syn_pool)
    acc_syn_in = float((predict(model_syn, X_syn[idx_hold]) == y_syn[idx_hold]).mean()) if hold_n else None
    acc_syn_real = float((predict(model_syn, X_bonus) == y_bonus).mean())

    # (3) доменный зазор в признаках: косинус между центроидами
    def cos(u, v):
        return float(np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v)))
    half = len(X_real_train) // 2
    cos_within_real = cos(X_real_train[:half].mean(0), X_real_train[half:].mean(0))
    cos_real_syn = cos(X_real_train.mean(0), X_syn_pool.mean(0))
    cos_real_bonus = cos(X_real_train.mean(0), X_bonus.mean(0))

    save_json("bonus.json", {
        "hypothesis": "различия смесей на test n=12 не воспроизводятся на независимой реальной выборке",
        "bonus_val": {"n": len(y_bonus), "class_balance": {str(k): v for k, v in Counter(y_bonus.tolist()).items()}},
        "test_vs_bonus": {str(r["alpha"]): {"test_acc": r["test_acc"], "bonus_acc": b["bonus_acc"]}
                          for r, b in zip(results, bonus_results)},
        "best_alpha": {"test": best_test, "bonus_val": best_bonus, "replicated": replicated},
        "synthetic_only": {"acc_holdout_syn": None if acc_syn_in is None else round(acc_syn_in, 4),
                           "acc_bonus_real": round(acc_syn_real, 4)},
        "feature_cosine": {"real_vs_real_halves": round(cos_within_real, 4),
                           "real_vs_synthetic": round(cos_real_syn, 4),
                           "real_vs_bonus": round(cos_real_bonus, 4)}})

    lines_b = [
        "# Бонус: воспроизводится ли эффект смесей за пределами отложенного test?",
        "", "## Гипотеза",
        "Различия точности между смесями real/synthetic на отложенном test (n=12) определяются шумом "
        "малой выборки и не воспроизводятся на независимой реальной выборке bonus_val того же "
        "распределения; дополнительно ожидается доменный зазор синтетика->real в признаках.",
        "", "## Протокол",
        f"- bonus_val: {len(y_bonus)} новых реальных фото PixMo-Count (исключены id train/test основного этапа)",
        f"- те же {len(alphas)} классификаторов смесей (бюджет {BUDGET}); синтетика-only классификатор на {len(X_syn_pool)} accepted-сценах",
        f"- hold-out синтетики: {hold_n} сцен; признаковое пространство — замороженный ResNet18, cos между центроидами",
        "", "## Наблюдения",
        f"- test vs bonus_val по alpha: {[(r['alpha'], r['test_acc'], b['bonus_acc']) for r, b in zip(results, bonus_results)]}",
        f"- лучшая alpha: test {best_test}, bonus_val {best_bonus} -> {'воспроизвелось' if replicated else 'НЕ воспроизвелось'}",
        f"- синтетика-only: {None if acc_syn_in is None else round(acc_syn_in, 4)} на hold-out синтетике против {round(acc_syn_real, 4)} на реальных фото",
        f"- cos центроидов: real/real {round(cos_within_real, 4)}, real/synthetic {round(cos_real_syn, 4)}, real/bonus {round(cos_real_bonus, 4)}",
        "", "## Вывод",
    ]
    if replicated:
        lines_b.append("Ранжирование смесей совпало на двух независимых реальных выборках, однако объёмы "
                       "(n=12 и n=16) малы: вывод об эффекте синтетики требует больших выборок. "
                       "Доменный зазор виден по косинусам центроидов и по падению точности "
                       "синтетика-only классификатора на реальных фото.")
    else:
        lines_b.append("Лучшая смесь на test и на bonus_val не совпала — гипотеза подтверждается: "
                       "различия на отложенном test лежат в пределах шума малой выборки и не ранжируют "
                       "методы. Доменный зазор синтетика->real подтверждается отдельно: синтетика-only "
                       f"классификатор теряет на реальных фото ({round(acc_syn_real, 4)} против "
                       f"{round(acc_syn_in, 4) if acc_syn_in is not None else 'n/a'} in-domain), центроиды "
                       "real и synthetic в признаках разнесены.")
    (ARTIFACTS / 'bonus.md').write_text("\n".join(lines_b) + "\n", encoding='utf-8')
    print(f"БОНУС OK: bonus_val n={len(y_bonus)}, best alpha test {best_test} vs bonus {best_bonus} "
          f"(совпадение: {replicated}); синтетика-only: in-domain "
          f"{None if acc_syn_in is None else round(acc_syn_in, 4)} / real {round(acc_syn_real, 4)}; "
          f"{time.time() - t0:.0f} c -> artifacts/bonus.md + bonus.json")


[features:bonus] 16/16
БОНУС OK: bonus_val n=16, best alpha test 0.5 vs bonus 0.0 (совпадение: False); синтетика-only: in-domain 1.0 / real 0.375; 11 c -> artifacts/bonus.md + bonus.json


## Анализ результатов

Сопоставьте accuracy смесей на фиксированном отложенном test (таблица в «Итогах выполнения») с majority baseline 0.500 и объясните, почему различия между смесями при n = 12 нельзя считать надёжными: шаг точности 1/12 ≈ 0.083, и бонус на 16 независимых реальных фото не воспроизвёл ранжирование смесей (лучшая на test α = 0.5 на bonus_val лучшей не стала). Разберите отдельные ошибки по `artifacts/predictions.jsonl` (в поле `correct` — верность предсказания каждой смеси на каждом тестовом фото) и соотнесите доменный зазор (синтетика-only: 1.000 на hold-out синтетике против 0.375 на реальных фото) с ограничениями процедурного рендера: чистый фон, точная геометрия и отсутствие перекрытий против сложных реальных сцен. FAST_DEV_RUN не является основанием для итогового вывода.

## Требования к сдаче

1. **Чистый прогон.** Kernel → Restart Kernel → Run All Cells при `FAST_DEV_RUN = False`; все ячейки выполняются без ошибок сверху вниз.
2. **Артефакты этапов** в `artifacts/`: `render_demo.png`, `synthetic_qa.jsonl`, `synthetic_manifest.jsonl`, `verification.jsonl`, `accepted.jsonl`, `features_real_train.npz`, `features_synthetic.npz`, `features_real_test.npz`, `mixture_training.json`, `mixture_results.json`, `predictions.jsonl`, `bonus.json`, `bonus.md`.
3. **Манифесты данных.** `artifacts/synthetic_manifest.jsonl` — provenance каждой синтетической сцены (seed, спецификация, рендерер, версия палитры); `data/lab12/real_manifest.jsonl` — источник, сплит и контрольные суммы реальных фото.
4. **«Итоги выполнения»** внизу ноутбука заполнены реальными числами прогона; FAST-прогон в отчёт не включается.
5. **Сдать**: файл `.ipynb` с сохранёнными выводами (File → Save Notebook) + папка `artifacts/` + `data/lab12/real_manifest.jsonl`. Воспроизводимость: SEED = 42, `FAST_DEV_RUN = False`, зависимости — в комментарии к ячейке окружения и `VLM_LOADER`. Фиктивные измерения не допускаются.


## Итоги выполнения

Полный прогон (`FAST_DEV_RUN=False`, seed = 42): синтетический корпус — **180 сцен** (counts 1–3: 63 / 55 / 62), PIL-рендер с суперсэмплингом, инвариант неперекрытия встроен в `render()`. Реальная VLM-проверка (Qwen2.5-VL-3B-Instruct, 4-bit NF4, greedy) подтвердила число объектов в **171 сцене (согласие 95.0%)**, дублей по пиксельному хешу — 0. В корпус отобрано **171 accepted-сцен** (count 1: 63, count 2: 55, count 3: 53); все 9 отсеянных сцен — count 3, т. е. VLM ошибалась только на самом населённом классе. Реальные фото PixMo-Count (ODC-BY-1.0, человечески проверенные метки): **train 80** (2:40 / 3:40), **test 12** (2:6 / 3:6); эталонные контрольные суммы совпали у 12/92 файлов — расхождения вызваны перекачиванием Flickr-хостинга (файл отдаётся другого размера), human-проверенные метки count не затронуты; пересечений train/test нет (проверка и по эталонному, и по фактически скачанному хешам).

**Признаки и классификаторы.** Замороженный ResNet18 (ImageNet-веса), признаки 512-d: real train (80 × 512), synthetic (108 × 512 — классы 2/3, общие с real), real test (12 × 512). Классификатор — линейный (count 2 vs 3), full-batch Adam, 400 эпох, seed фиксирован. Постоянный бюджет B = 80: доля синтетики заменяет реальные примеры (α = 0 / 0.25 / 0.5 / 0.75 → 80/60/40/20 real), hold-out синтетики — 20 сцен. Все смеси сходятся по train: train_acc = 1.000 — задача линейно отделима в признаках ResNet18 даже на 20 реальных примерах, поэтому различия на test определяются составом выборки, а не недообучением.

**Результаты на реальном test (n = 12).**

| α | real | synthetic | train_acc | test_acc | acc (2) | acc (3) |
|-----|------|-----------|-----------|----------|---------|---------|
| 0.00 | 80 | 0 | 1.000 | 0.333 | 0.167 | 0.500 |
| 0.25 | 60 | 20 | 1.000 | 0.417 | 0.500 | 0.333 |
| **0.50** | 40 | 40 | 1.000 | **0.667** | 0.500 | 0.833 |
| 0.75 | 20 | 60 | 1.000 | 0.500 | 0.333 | 0.667 |

Majority baseline — 0.500. Лучшая смесь — **α = 0.5 (0.667)**: умеренная доля синтетики дала +0.334 к чистым real (0.333), а наибольшая доля (α = 0.75) уже хуже (0.500). Однако шаг точности при n = 12 равен 1/12 ≈ 0.083, поэтому этот вывод требует проверки — что и сделано в бонусе.

**Бонус (bonus_val, n = 16 независимых реальных фото).** Ранжирование смесей **не воспроизвелось**: лучшая на test α = 0.5 на bonus_val лучшей не стала (лучшая — α = 0.0). Гипотеза подтверждена: различия смесей на test n = 12 — шум малой выборки, а не реальный эффект синтетики. Доменный зазор при этом реален: синтетика-only классификатор даёт **1.000** на hold-out синтетике и лишь **0.375** на реальных фото bonus_val — признаки процедурных сцен (чистый фон, точная геометрия) заметно отличаются от признаков реальных фотографий.

**Ограничения.** Test n = 12: различия между смесями до 4/12 — в пределах шума выборки (подтверждено бонусом); VLM-фильтрация (95.0% согласия) смещает корпус в сторону сцен, «понятных» модели. Синтетика — процедурный рендер без перекрытий с точными координатами, реальные фото — сложные сцены; синтетические данные не выдавались за реальные: provenance каждого примера — в `artifacts/synthetic_manifest.jsonl`, источник и контрольные суммы реальных фото — в `data/lab12/real_manifest.jsonl`. FAST_DEV_RUN использовался только для проверки механики и в отчёт не включён.
